# Retrieval Evaluation on SciFact (BeIR)

This notebook builds and evaluates three retrieval systems on the same SciFact corpus and query set:

- **BM25** (keyword retrieval)
- **Dense retrieval** using `sentence-transformers` + FAISS
- **Hybrid retrieval** with score fusion

## Task

Given a query, retrieve the most relevant scientific document(s) from the SciFact corpus.

## Dataset

We use Hugging Face dataset `BeIR/scifact`:

- `corpus`: documents to retrieve
- `queries`: user/test queries
- `qrels`: relevance labels (ground truth)

Each document is treated as one retrievable unit.

## Evaluation Goals

We compare all three systems on identical queries using:

- Recall@5
- MRR
- p95 latency (ms)

In [ ]:
import re
import time
from collections import defaultdict

import faiss
import numpy as np
import pandas as pd
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

pd.set_option("display.max_colwidth", 120)

## 2) Load and Inspect Dataset

This section loads `corpus`, `queries`, and `qrels`, prints sample entries, and builds the core mappings needed for retrieval/evaluation.

In [ ]:
def load_scifact_splits():
    """Load SciFact corpus/queries/qrels robustly across dataset API variants."""
    try:
        ds = load_dataset("BeIR/scifact")
        if all(k in ds for k in ["corpus", "queries", "qrels"]):
            corpus = ds["corpus"]
            queries = ds["queries"]
            qrels = ds["qrels"]
            if hasattr(qrels, "keys") and "test" in qrels:
                qrels = qrels["test"]
            return corpus, queries, qrels
    except Exception:
        pass

    corpus = load_dataset("BeIR/scifact", "corpus", split="corpus")
    queries = load_dataset("BeIR/scifact", "queries", split="queries")
    qrels = load_dataset("BeIR/scifact", "qrels", split="test")
    return corpus, queries, qrels


corpus_ds, queries_ds, qrels_ds = load_scifact_splits()

print("Corpus size:", len(corpus_ds))
print("Queries size:", len(queries_ds))
print("Qrels size:", len(qrels_ds))

print("\nSample corpus entry:")
print(corpus_ds[0])
print("\nSample query entry:")
print(queries_ds[0])
print("\nSample qrels entry:")
print(qrels_ds[0])

In [ ]:
def coalesce_text(title, text):
    title = title or ""
    text = text or ""
    combined = (title + " " + text).strip()
    return re.sub(r"\s+", " ", combined)


def clean_text(s):
    s = "" if s is None else str(s)
    s = s.replace("\n", " ").replace("\t", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


# Build document and query mappings
corpus_rows = list(corpus_ds)
query_rows = list(queries_ds)
qrels_rows = list(qrels_ds)

doc_id_to_text = {}
doc_id_to_title = {}

for row in corpus_rows:
    doc_id = str(row.get("_id", row.get("doc_id", row.get("id"))))
    title = clean_text(row.get("title", ""))
    text = clean_text(row.get("text", ""))
    doc_id_to_title[doc_id] = title
    doc_id_to_text[doc_id] = coalesce_text(title, text)

query_id_to_text = {}
for row in query_rows:
    qid = str(row.get("_id", row.get("query_id", row.get("id"))))
    qtext = clean_text(row.get("text", row.get("query", "")))
    query_id_to_text[qid] = qtext

print("Unique docs:", len(doc_id_to_text))
print("Unique queries:", len(query_id_to_text))